# Lesson 7 — Combining tables
**Goal:** real data lives in several tables. Join orders ↔ products ↔ customers, and match mobile-money payments to orders to find problems.

In [1]:
import pandas as pd
orders    = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
products  = pd.read_csv('data/products.csv')
customers = pd.read_csv('data/customers.csv', parse_dates=['signup_date'])
mm        = pd.read_csv('data/mobile_money.csv', parse_dates=['timestamp'])

## `merge` = SQL JOIN / Excel VLOOKUP

In [2]:
full = (orders
        .merge(products[['product_id', 'product_name', 'category', 'unit_cost']], on='product_id', how='left')
        .merge(customers[['customer_id', 'name', 'city', 'segment']], on='customer_id', how='left'))
full[['order_id', 'product_name', 'name', 'city', 'segment']].head()

,order_id,product_name,name,city,segment
0,O00001,Wireless Mouse,Juma Mwakyusa,Dar es Salaam,Small Business
1,O00002,HDMI Cable 2m,Agnes Nyerere,Arusha,Individual
2,O00003,Wireless Mouse,Peter Juma,Dar es Salaam,Individual
3,O00004,USB-C Charger 65W,Peter Nyerere,Arusha,Individual
4,O00005,HDMI Cable 2m,Emmanuel Juma,Arusha,Individual


In [3]:
full.shape, orders.shape   # a left join keeps every order

((2400, 16), (2400, 10))

## Which customer segment spends most?

In [4]:
full['revenue'] = full['quantity'] * full['unit_price'] * (1 - full['discount_pct']/100)
full.groupby('segment')['revenue'].agg(['count', 'sum', 'mean']).round(0)

,count,sum,mean
segment,,,
Individual,1315,171800800.0,130647.0
Small Business,705,89572650.0,127053.0
Student,380,58260850.0,153318.0


## Join types matter: reconciling mobile-money payments
**Question:** every Mobile Money order should have a successful payment. Which don't?

In [5]:
mm_orders = orders[orders['payment_method'] == 'Mobile Money']
check = mm_orders.merge(mm, on='order_id', how='left', indicator=True)
check['_merge'].value_counts()   # left_only = order with NO payment record

_merge
both          1362
left_only       20
right_only       0
Name: count, dtype: int64

In [6]:
no_payment = check[check['_merge'] == 'left_only']
print(len(no_payment), 'Mobile Money orders have no payment at all')

20 Mobile Money orders have no payment at all


In [7]:
check['status'].value_counts(dropna=False)

status
SUCCESS     1136
FAILED       124
REVERSED     102
NaN           20
Name: count, dtype: int64

In [8]:
problems = check[check['status'] != 'SUCCESS'][['order_id', 'order_date', 'branch', 'provider', 'amount_tzs', 'status']]
print(len(problems), 'mobile-money orders need follow-up')
problems.head()

246 mobile-money orders need follow-up


,order_id,order_date,branch,provider,amount_tzs,status
12,O00028,2025-01-07,Mwanza Rock City,Tigo Pesa,320000.0,REVERSED
17,O00041,2025-01-10,Mlimani City,M-Pesa,55000.0,FAILED
19,O00044,2025-01-11,Mlimani City,Airtel Money,18000.0,REVERSED
22,O00049,2025-01-11,Mlimani City,NaN,NaN,NaN
29,O00060,2025-01-13,Kariakoo,Tigo Pesa,45000.0,REVERSED


In [9]:
problems.groupby('provider')['amount_tzs'].agg(['count', 'sum'])

,count,sum
provider,,
Airtel Money,46,4741600.0
HaloPesa,13,908300.0
M-Pesa,117,14192700.0
Tigo Pesa,50,8173300.0


## `concat` — stack tables with the same columns
E.g. monthly exports into one year:

In [10]:
jan = orders[orders['order_date'].dt.month == 1]
feb = orders[orders['order_date'].dt.month == 2]
q = pd.concat([jan, feb], ignore_index=True)
len(jan), len(feb), len(q)

(153, 120, 273)

## Key takeaways
- `merge(on=..., how='left'|'inner'|'outer')` joins tables on a key; `indicator=True` shows what matched.
- Reconciliation (orders vs payments) is a classic real-world merge job.
- `concat` stacks tables vertically.

**Exercise:** which city's customers bring the most revenue per customer?